In [1]:
# Cell 1: Configuration and imports

import json
import os
import smtplib
import time

from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from typing import List

import gradio as gr
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError, field_validator


load_dotenv(".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.7

EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587

DAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

INGREDIENTS_PER_RECIPE = 5

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %% Cell 1b: Error helpers + config check  (NEW, put right after Cell 1)
 
import openai
import smtplib
from typing import Literal
 
 
class ConfigError(Exception):
    """Required settings are missing."""
 
 
class ResponseParseError(Exception):
    """The LLM response could not be turned into JSON."""
 
 
class TestDataError(Exception):
    """A test-case file is missing, malformed or invalid."""
 
 
def format_error(function_name: str, error: Exception,
                 context: str = "", suggestion: str = "") -> str:
    """One format for every error: where, what type, context, what to do."""
    lines = [
        f"[ERROR] {function_name} | {type(error).__name__}",
        f"Message: {error}",
    ]
    if context:
        lines.append(f"Context: {context}")
    if suggestion:
        lines.append(f"Suggestion: {suggestion}")
    return "\n".join(lines)
 
 
def describe_validation_error(error) -> str:
    """Turn a Pydantic ValidationError into 'field.path: message' lines."""
    return "; ".join(
        f"{'.'.join(str(part) for part in item['loc'])}: {item['msg']}"
        for item in error.errors()
    )
 
 
def default_if_blank(value, default: str) -> str:
    """Return default for None, empty or whitespace-only input."""
    if value is None:
        return default
    value = str(value).strip()
    return value or default

In [3]:
# Fail early and clearly instead of crashing inside ChatOpenAI()
if not OPENAI_API_KEY:
    raise ConfigError(format_error(
        "config",
        ConfigError("OPENAI_API_KEY is not set"),
        context="Looked in environment and .env",
        suggestion="Add OPENAI_API_KEY=sk-... to .env next to this notebook.",
    ))

In [4]:
# Cell 2: Data models

class Ingredient(BaseModel):
    name: str
    quantity: str


class Recipe(BaseModel):
    day: str
    theme: str
    dish_name: str
    ingredients: List[Ingredient]
    instructions: str

    @field_validator("ingredients")
    @classmethod
    def validate_ingredient_count(cls, ingredients):
        if len(ingredients) != INGREDIENTS_PER_RECIPE:
            raise ValueError(
                f"Recipe must contain exactly "
                f"{INGREDIENTS_PER_RECIPE} ingredients."
            )
        return ingredients


class ShoppingItem(BaseModel):
    name: str
    total_quantity: str
    used_in: List[str]


class MealPlan(BaseModel):
    dietary_preference: str
    recipes: List[Recipe]
    shopping_list: List[ShoppingItem]

    @field_validator("recipes")
    @classmethod
    def validate_recipe_count(cls, recipes):
        expected_count = len(DAYS)

        if len(recipes) != expected_count:
            raise ValueError(
                f"Meal plan must contain exactly "
                f"{expected_count} recipes."
            )
        return recipes

    @field_validator("recipes")
    @classmethod
    def validate_unique_themes(cls, recipes):
        themes = [recipe.theme.lower() for recipe in recipes]

        if len(themes) != len(set(themes)):
            raise ValueError("All recipe themes must be unique.")

        return recipes

In [5]:
# Cell 3: LLM setup and prompt

def get_llm(api_key: str = None, base_url: str = None, max_retries: int = 2):
    """Parameters exist so tests can inject a fake key or unreachable server."""
    return ChatOpenAI(
        api_key=api_key or OPENAI_API_KEY,
        base_url=base_url,
        model=OPENAI_MODEL,
        temperature=TEMPERATURE,
        timeout=30,
        max_retries=max_retries,
    )


MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional meal planner.

Generate a weekly meal plan with exactly 7 themed dinners.

Rules:
- Generate exactly 7 recipes.
- Use a different cuisine theme for every recipe.
- Each recipe must contain exactly 5 main ingredients.
- Follow the dietary preference strictly.
- Provide realistic quantities for 2 servings.
- Return valid JSON only.

The JSON must match this structure:
{format_instructions}
"""
    ),
    (
        "human",
        """Create a weekly meal plan.

Dietary preference: {dietary_preference}
Theme requests: {theme_requests}
Additional preferences: {additional_preferences}
"""
    ),
])

In [6]:
def extract_json(text: str) -> dict:
    """Extract the first JSON object from an LLM response."""
    cleaned_text = text.replace("```json", "").replace("```", "").strip()
    start_index = cleaned_text.find("{")
 
    if start_index == -1:
        raise ResponseParseError(
            f"No JSON object found. Response starts with: "
            f"{cleaned_text[:80]!r}"
        )
 
    try:
        data, _ = json.JSONDecoder().raw_decode(cleaned_text[start_index:])
    except json.JSONDecodeError as error:
        # Keep the position info instead of throwing it away
        raise ResponseParseError(
            f"Invalid JSON at line {error.lineno}, column {error.colno}: "
            f"{error.msg}"
        ) from error
 
    return data
 
 
class MealPlanGenerator:
    """Generates and validates weekly meal plans."""
 
    def __init__(self, llm=None):
        self.llm = llm or get_llm()
        self.parser = JsonOutputParser(pydantic_object=MealPlan)
 
    def _build_request(self, dietary_preference, theme_requests,
                       additional_preferences) -> dict:
        return {
            "dietary_preference": dietary_preference,
            "theme_requests": default_if_blank(
                theme_requests, "random diverse cuisines"),
            "additional_preferences": default_if_blank(
                additional_preferences, "none"),
            "format_instructions": self.parser.get_format_instructions(),
        }
 
    def _call_llm(self, request: dict):
        chain = MEAL_PLAN_PROMPT | self.llm
        return chain.invoke(request)
 
    def _parse_response(self, response) -> MealPlan:
        data = extract_json(response.content)
        return MealPlan(**data)
 
    @staticmethod
    def _result(start_time, meal_plan=None, error=None) -> dict:
        """Success and failure always have the same keys."""
        return {
            "success": error is None,
            "meal_plan": meal_plan,
            "generation_time": round(time.time() - start_time, 2),
            "error": error,
        }
 
    def generate(self, dietary_preference: str,
                 theme_requests: str = "random",
                 additional_preferences: str = "none") -> dict:
        """Generate one validated meal plan."""
        start_time = time.time()
        context = f"diet={dietary_preference!r}, model={OPENAI_MODEL}"
        api = "MealPlanGenerator._call_llm"
        parse = "MealPlanGenerator._parse_response"
 
        try:
            request = self._build_request(
                dietary_preference, theme_requests, additional_preferences)
            response = self._call_llm(request)
            meal_plan = self._parse_response(response)
            return self._result(start_time, meal_plan=meal_plan.model_dump())
 
        # --- API errors: most specific first (subclasses before parents) ---
        except openai.AuthenticationError as error:
            message = format_error(api, error, context,
                "Check OPENAI_API_KEY in .env. Retrying will not help.")
        except openai.RateLimitError as error:
            message = format_error(api, error, context,
                "Rate limit or quota reached. Wait a minute, or check billing.")
        except openai.APITimeoutError as error:
            message = format_error(api, error, f"{context}, timeout=30s",
                "The API did not answer in time. Try again.")
        except openai.APIConnectionError as error:
            message = format_error(api, error, context,
                "Could not reach OpenAI. Check your internet connection.")
        except openai.APIStatusError as error:
            message = format_error(api, error,
                f"{context}, status={error.status_code}",
                "OpenAI returned an error status. Check model name and status page.")
        except openai.APIError as error:
            message = format_error(api, error, context,
                "Unexpected OpenAI error. Try again later.")
 
        # --- Response errors ---
        except ResponseParseError as error:
            message = format_error(parse, error, context,
                "The model did not return clean JSON. Run again; if it repeats, "
                "tighten the 'Return valid JSON only' rule in the prompt.")
        except ValidationError as error:
            # Caught explicitly: in Pydantic v2 this is a ValueError subclass
            message = format_error(f"{parse} -> MealPlan", error,
                f"{context}, fields: {describe_validation_error(error)}",
                "The model broke a schema rule. Run again or strengthen the prompt.")
 
        # No catch-all: a genuine bug should stop and show a full traceback
        return self._result(start_time, error=message)
 
 
generator = MealPlanGenerator()
 

In [7]:
# Cell 5: Validation  
 
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Reuse Pydantic validation instead of duplicating rules."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}
 
    try:
        MealPlan(**meal_plan_result["meal_plan"])
        return {"passed": True, "errors": []}
    except ValidationError as error:
        return {
            "passed": False,
            "errors": [format_error(
                "run_basic_validation -> MealPlan", error,
                describe_validation_error(error),
                "Fix the listed fields.",
            )],
        }

In [8]:
# Cell 6: Formatting helpers

def format_ingredients(ingredients: list) -> str:
    """Convert ingredients into readable text."""

    return ", ".join(
        f"{ingredient['quantity']} {ingredient['name']}"
        for ingredient in ingredients
    )


def format_shopping_list(meal_plan: dict) -> str:
    lines = ["## Shopping List\n"]

    for item in meal_plan["shopping_list"]:
        lines.append(
            f"- {item['total_quantity']} {item['name']}"
        )

    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    ingredients = format_ingredients(
        recipe["ingredients"]
    )

    return (
        f"**{recipe['day']}** | "
        f"{recipe['theme']} | "
        f"*{recipe['dish_name']}*\n"
        f"{ingredients}\n"
    )

In [9]:
# Cell 7: Email formatting and sending separated

def format_meal_plan_email(meal_plan: dict) -> str:
    """Build the readable email body."""

    lines = [
        "=" * 40,
        f"WEEKLY MEAL PLAN "
        f"({meal_plan['dietary_preference'].upper()})",
        "=" * 40,
        "",
        "SHOPPING LIST",
        "-" * 20,
    ]

    for item in meal_plan["shopping_list"]:
        lines.append(
            f"- {item['total_quantity']} {item['name']}"
        )

    lines.extend([
        "",
        "=" * 40,
        "RECIPES",
        "=" * 40,
    ])

    for recipe in meal_plan["recipes"]:
        lines.extend([
            f"\n{recipe['day'].upper()} - {recipe['theme']}",
            recipe["dish_name"],
            f"Ingredients: "
            f"{format_ingredients(recipe['ingredients'])}",
            f"Instructions: {recipe['instructions']}",
        ])

    return "\n".join(lines)


def build_email_message(
    recipient_email: str,
    meal_plan: dict,
) -> MIMEMultipart:
    """Build an email message without sending it."""

    message = MIMEMultipart()
    message["From"] = EMAIL_SENDER
    message["To"] = recipient_email
    message["Subject"] = "Your Weekly Meal Plan"

    body = format_meal_plan_email(meal_plan)
    message.attach(MIMEText(body, "plain"))

    return message

def send_email_message(message) -> None:
    """Send an already-built message over SMTP."""
    with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=15) as server:
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        

In [10]:
# Cell 8: UI functions

def generate_plan_streaming(
    dietary_preference,
    theme_requests,
    additional_preferences,):
    """Generate one plan and store that same plan for emailing."""

    yield "Generating your meal plan...", None

    result = generator.generate(
        dietary_preference=dietary_preference,
        theme_requests=theme_requests,
        additional_preferences=additional_preferences,
    )

    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]

    output = (
        f"## Weekly "
        f"{meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    )
    output += format_shopping_list(meal_plan)
    output += "\n\n---\n\n## Recipes\n\n"

    yield output, meal_plan

    for recipe in meal_plan["recipes"]:
        output += format_single_recipe(recipe)
        output += "\n"
        yield output, meal_plan

    validation = run_basic_validation(result)

    status = (
        "All checks passed"
        if validation["passed"]
        else f"Issues: {', '.join(validation['errors'])}"
    )

    output += (
        f"---\n"
        f"*Generated in {result['generation_time']}s | "
        f"{status}*"
    )

    yield output, meal_plan



def send_email_ui(email: str, meal_plan: dict):
    """Email the currently displayed plan."""
    if not email or "@" not in email:
        return "Enter a valid email address."
    if not meal_plan:
        return "Generate a meal plan before emailing it."
    if not EMAIL_SENDER or not EMAIL_PASSWORD:
        return format_error("send_email_ui",
            ConfigError("Email credentials missing"),
            "EMAIL_SENDER / EMAIL_PASSWORD not set",
            "Add both to .env (use a Gmail App Password).")
 
    server = f"{SMTP_HOST}:{SMTP_PORT}"
    try:
        message = build_email_message(email, meal_plan)
        send_email_message(message)
        return f"Email sent to {email}"

    except Exception as error:
        return f"Email sending failed: {error}"

In [11]:
# Cell 9: Gradio interface

with gr.Blocks(title="Weekly Meal Planner") as app:
    meal_plan_state = gr.State(value=None)

    gr.Markdown(
        "# Weekly Meal Planner\n"
        "Generate 7 themed dinners with a shopping list."
    )

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=[
                    "omnivore",
                    "vegetarian",
                    "vegan",
                    "keto",
                ],
                value="omnivore",
                label="Diet",
            )

            theme_input = gr.Textbox(
                label="Themes",
                placeholder="Italian, Mexican...",
            )

            additional_input = gr.Textbox(
                label="Additional preferences",
                placeholder="No mushrooms, high protein...",
            )

            generate_button = gr.Button(
                "Generate",
                variant="primary",
            )

            email_input = gr.Textbox(
                label="Email",
                placeholder="you@example.com",
            )

            send_button = gr.Button("Email Plan")

            email_status = gr.Textbox(
                label="Status",
                interactive=False,
            )

        with gr.Column(scale=2):
            output_display = gr.Markdown(
                "Click **Generate** to start."
            )

    generate_button.click(
        fn=generate_plan_streaming,
        inputs=[
            dietary_dropdown,
            theme_input,
            additional_input,
        ],
        outputs=[
            output_display,
            meal_plan_state,
        ],
    )

    send_button.click(
        fn=send_email_ui,
        inputs=[
            email_input,
            meal_plan_state,
        ],
        outputs=email_status,
    )

In [12]:
#cell 10a: Test-case loading 
 
class TestCase(BaseModel):
    dietary_preference: Literal["omnivore", "vegetarian", "vegan", "keto"]
    theme_requests: str = "random"
    additional_preferences: str = "none"
 
 
def load_test_cases(path: str) -> list:
    """Load and validate test cases from a JSON file."""
    name = "load_test_cases"
    try:
        with open(path, encoding="utf-8") as file:
            raw_cases = json.load(file)
    except FileNotFoundError as error:
        raise TestDataError(format_error(name, error, f"path={path!r}",
            "Check the file name and that it sits next to the notebook.")) from error
    except json.JSONDecodeError as error:
        raise TestDataError(format_error(name, error,
            f"path={path!r}, line {error.lineno}, column {error.colno}",
            "Fix the JSON syntax at that position.")) from error
 
    if not isinstance(raw_cases, list):
        raise TestDataError(format_error(name,
            TypeError(f"expected a list, got {type(raw_cases).__name__}"),
            f"path={path!r}", "Wrap the test cases in [ ... ]."))
 
    cases = []
    for index, raw_case in enumerate(raw_cases):
        try:
            cases.append(TestCase(**raw_case).model_dump())
        except (ValidationError, TypeError) as error:
            detail = (describe_validation_error(error)
                      if isinstance(error, ValidationError) else str(error))
            raise TestDataError(format_error(name, error,
                f"path={path!r}, case #{index}: {detail}",
                "Fix that test case.")) from error
    return cases
 

In [13]:
# Cell 10: External test data and test runner

def run_test_cases(test_cases: list) -> list:
    """Run test cases and return structured results."""

    results = []

    for case in test_cases:
        result = generator.generate(**case)
        validation = run_basic_validation(result)

        results.append({
            "dietary_preference": case["dietary_preference"],
            "passed": validation["passed"],
            "errors": validation["errors"],
        })

    return results


try:
    test_cases = load_test_cases("test_cases.json")
except TestDataError as error:
    print(error)
    test_cases = []

for test in run_test_cases(test_cases):
    status = "PASS" if test["passed"] else "FAIL\n" + "\n".join(test["errors"])
    print(f"{test['dietary_preference']} -> {status}\n")

vegetarian -> FAIL
[ERROR] MealPlanGenerator._parse_response -> MealPlan | ValidationError
Message: 1 validation error for MealPlan
recipes
  Value error, All recipe themes must be unique. [type=value_error, input_value=[{'day': 'Monday', 'theme...hini sauce and serve.'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
Context: diet='vegetarian', model=gpt-4o-mini, fields: recipes: Value error, All recipe themes must be unique.
Suggestion: The model broke a schema rule. Run again or strengthen the prompt.

vegan -> PASS

omnivore -> PASS



In [14]:
from pathlib import Path
 
Path("malformed.json").write_text(
    '[{"dietary_preference": "vegan", "theme_requests": "random",}]')
Path("invalid_test_cases.json").write_text(json.dumps([
    {"dietary_preference": "vegan"},
    {"dietary_preference": "carnivore", "theme_requests": "random"},
]))
 
 
def show(label: str, text: str) -> None:
    print(f"===== {label} =====\n{text}\n")
 
 
def try_load(path: str) -> str:
    try:
        load_test_cases(path)
        return "Loaded without error (unexpected)"
    except TestDataError as error:
        return str(error)
 
 
def try_parse(text: str) -> str:
    try:
        extract_json(text)
        return "Parsed without error"
    except ResponseParseError as error:
        return format_error("extract_json", error)
 
 
def make_sample_plan(recipe_count=7, ingredient_count=5, duplicate_theme=False):
    themes = ["Italian", "Mexican", "Japanese", "Indian",
              "Thai", "Greek", "French", "Korean"]
    return {
        "dietary_preference": "omnivore",
        "shopping_list": [],
        "recipes": [
            {
                "day": DAYS[i % len(DAYS)],
                "theme": "Italian" if duplicate_theme else themes[i],
                "dish_name": f"Dish {i}",
                "ingredients": [{"name": f"item {j}", "quantity": "100 g"}
                                for j in range(ingredient_count)],
                "instructions": "Cook.",
            }
            for i in range(recipe_count)
        ],
    }
 
 
def try_plan(plan: dict) -> str:
    result = run_basic_validation({"success": True, "meal_plan": plan})
    return "Valid" if result["passed"] else "\n".join(result["errors"])


In [15]:
# Cell 11: Error-scenario tests  (NEW, no paid API calls)
# Each section is one screenshot for the submission.
 
from pathlib import Path
 
Path("malformed.json").write_text(
    '[{"dietary_preference": "vegan", "theme_requests": "random",}]')
Path("invalid_test_cases.json").write_text(json.dumps([
    {"dietary_preference": "vegan"},
    {"dietary_preference": "carnivore", "theme_requests": "random"},
]))
 
 
def show(label: str, text: str) -> None:
    print(f"===== {label} =====\n{text}\n")
 
 
def try_load(path: str) -> str:
    try:
        load_test_cases(path)
        return "Loaded without error (unexpected)"
    except TestDataError as error:
        return str(error)
 
 
def try_parse(text: str) -> str:
    try:
        extract_json(text)
        return "Parsed without error"
    except ResponseParseError as error:
        return format_error("extract_json", error)
 
 
def make_sample_plan(recipe_count=7, ingredient_count=5, duplicate_theme=False):
    themes = ["Italian", "Mexican", "Japanese", "Indian",
              "Thai", "Greek", "French", "Korean"]
    return {
        "dietary_preference": "omnivore",
        "shopping_list": [],
        "recipes": [
            {
                "day": DAYS[i % len(DAYS)],
                "theme": "Italian" if duplicate_theme else themes[i],
                "dish_name": f"Dish {i}",
                "ingredients": [{"name": f"item {j}", "quantity": "100 g"}
                                for j in range(ingredient_count)],
                "instructions": "Cook.",
            }
            for i in range(recipe_count)
        ],
    }
 
 
def try_plan(plan: dict) -> str:
    result = run_basic_validation({"success": True, "meal_plan": plan})
    return "Valid" if result["passed"] else "\n".join(result["errors"])
 
 
# 1. Files
show("Missing file", try_load("does_not_exist.json"))
show("Malformed JSON file", try_load("malformed.json"))
show("Invalid test data", try_load("invalid_test_cases.json"))
 
# 2. LLM response parsing
show("Text before JSON", try_parse('Sure! Here it is: {"recipes": []}'))
show("No JSON at all", try_parse("Sorry, I cannot help with that."))
show("Broken JSON", try_parse('{"recipes": [}'))
 
# 3. Schema validation
show("Valid sample plan", try_plan(make_sample_plan()))
show("6 recipes", try_plan(make_sample_plan(recipe_count=6)))
show("4 ingredients", try_plan(make_sample_plan(ingredient_count=4)))
show("Duplicate themes", try_plan(make_sample_plan(duplicate_theme=True)))
 
# 4. Empty / None input (should not crash)
show("None textbox", json.dumps(
    generator._build_request("vegan", None, "   "), indent=2)[:300])
 
# 5. API errors without spending money
fake_key = MealPlanGenerator(llm=get_llm(api_key="sk-fake-key", max_retries=0))
show("Wrong API key", fake_key.generate("vegan")["error"])
 
no_network = MealPlanGenerator(llm=get_llm(
    base_url="http://127.0.0.1:9", max_retries=0))
show("Network unreachable", no_network.generate("vegan")["error"])
 

===== Missing file =====
[ERROR] load_test_cases | FileNotFoundError
Message: [Errno 2] No such file or directory: 'does_not_exist.json'
Context: path='does_not_exist.json'
Suggestion: Check the file name and that it sits next to the notebook.

===== Malformed JSON file =====
[ERROR] load_test_cases | JSONDecodeError
Message: Expecting property name enclosed in double quotes: line 1 column 61 (char 60)
Context: path='malformed.json', line 1, column 61
Suggestion: Fix the JSON syntax at that position.

===== Invalid test data =====
[ERROR] load_test_cases | ValidationError
Message: 1 validation error for TestCase
dietary_preference
  Input should be 'omnivore', 'vegetarian', 'vegan' or 'keto' [type=literal_error, input_value='carnivore', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Context: path='invalid_test_cases.json', case #1: dietary_preference: Input should be 'omnivore', 'vegetarian', 'vegan' or 'keto'
Suggestion: Fix that tes